# 🏆 Task 4: SOTA ConvNeXt-V2 End-to-End Fine-Tuning on Google Colab (T4 / A100 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Palapluem/ml00x/blob/main/task4/notebooks/colab_gpu_dinov2_training.ipynb)

This notebook provides the **official 1st-place architecture** (`convnextv2_tiny.fcmae_ft_in22k_in1k`) adapted from `Saltywan/CPE342-KAGGLE` for **Task 4: Game Title Classification**.

### Key Architectural Features:
1. **Direct Google Drive High-Speed Download:** Automates downloading the official dataset archive (`task4_image_classification.zip`, File ID: `1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw`) at 100+ MB/s.
2. **Aspect-Preserving Widescreen Geometry:** Uses $128 \times 256$ resolution without squashing gameplay HUD elements.
3. **Photometric Darkness Shift Compensation:** Augments train images with brightness/contrast shifts to bridge the $-8.3\%$ test darkness gap.
4. **5-Fold Stratified Cross-Validation:** Full end-to-end backbone fine-tuning with CosineAnnealingLR and Class-Weighted Loss.
5. **4x Test-Time Augmentation (TTA) + Macro-$F_1$ Calibration:** Guarantees balanced minority-class recall and outputs official integer labels `0..4`.

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q timm gdown torchvision scikit-learn pandas numpy matplotlib albumentations tqdm

In [ ]:
# 2. High-Speed Download & Dataset Extraction
import os, zipfile
from pathlib import Path

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Check if dataset is already extracted
if not (DATA_DIR / 'train.csv').exists():
    print('Downloading Task 4 dataset from Google Drive...')
    !gdown --id 1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw -O task4_image_classification.zip
    print('Extracting archive...')
    with zipfile.ZipFile('task4_image_classification.zip', 'r') as zip_ref:
        zip_ref.extractall('data/')
    print('Extraction complete!')
else:
    print('Dataset already present in data/!')

# Verify file locations
!ls -la data/

In [ ]:
# 3. Imports & Configuration
import os, sys, time, random, copy
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

ImageFile.LOAD_TRUNCATED_IMAGES = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')

# Official Competition Mapping (datacard.txt):
# 0: 7-11 Knights ('7-11k' / '7-11K')
# 1: FiveN ('fiveN')
# 2: Free Fried ('freefried')
# 3: PUBE ('pube')
# 4: Roblock ('roblock')
CLASSES = ['7-11K', 'fiveN', 'freefried', 'pube', 'roblock']
c2i = {c: i for i, c in enumerate(CLASSES)}
c2i['7-11k'] = 0

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()

In [ ]:
# 4. Widescreen Dataset with Darkness Compensation Augmentations
class GameDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id']
        # Handle nested or flat folders
        img_path = self.img_dir / f'{img_id}.jpg'
        if not img_path.exists():
            # search fallback
            img_path = next(self.img_dir.glob(f'**/{img_id}.jpg'))
        
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, img_id
        label = c2i[row['class'].strip()]
        return img, label

# Photometric training transform:
# Compensates for the -8.3% darkness shift observed on test screenshots
train_transform = transforms.Compose([
    transforms.Resize((128, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.20, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((128, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 4x Test-Time Augmentations (TTA)
tta_transforms = [
    val_transform,
    transforms.Compose([transforms.Resize((128, 256)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((128, 256)), transforms.ColorJitter(brightness=(1.08, 1.08)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((128, 256)), transforms.RandomHorizontalFlip(p=1.0), transforms.ColorJitter(brightness=(1.08, 1.08)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
]

In [ ]:
# 5. SOTA ConvNeXt-V2 Model Builder
def build_model(backbone_name='convnextv2_tiny.fcmae_ft_in22k_in1k', num_classes=5):
    model = timm.create_model(backbone_name, pretrained=True, num_classes=num_classes)
    return model.to(device)

In [ ]:
# 6. 5-Fold Stratified Training Pipeline with CosineAnnealingLR
train_csv_path = DATA_DIR / 'train.csv'
test_csv_path = DATA_DIR / 'test.csv'

train_df = pd.read_csv(train_csv_path)
train_df['class'] = train_df['class'].str.strip()
test_df = pd.read_csv(test_csv_path)

# Locate train and test image folders
train_img_dir = DATA_DIR / 'train' if (DATA_DIR / 'train').exists() else DATA_DIR
test_img_dir = DATA_DIR / 'test' if (DATA_DIR / 'test').exists() else DATA_DIR

y_all = np.array([c2i[c] for c in train_df['class']], dtype=np.int64)
counts = np.bincount(y_all, minlength=5)
class_weights = torch.tensor(len(y_all) / (5.0 * counts), dtype=torch.float32, device=device)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probs = np.zeros((len(train_df), 5), dtype=np.float32)
test_probs = np.zeros((len(test_df), 5), dtype=np.float32)

EPOCHS = 6
BATCH_SIZE = 64
LR = 1e-4

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
    print(f'\n========== Fold {fold}/5 ==========')
    tr_sub = train_df.iloc[tr_idx]
    va_sub = train_df.iloc[va_idx]
    
    ds_tr = GameDataset(tr_sub, train_img_dir, transform=train_transform)
    ds_va = GameDataset(va_sub, train_img_dir, transform=val_transform)
    
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    
    model = build_model()
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.03)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(dl_tr), eta_min=1e-6)
    
    best_f1 = -1.0
    best_state = None
    
    for ep in range(1, EPOCHS + 1):
        model.train()
        tr_loss = 0.0
        for x, y in tqdm(dl_tr, desc=f'Fold {fold} Ep {ep}/{EPOCHS}'):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            scheduler.step()
            tr_loss += loss.item() * len(y)
        
        # Validation
        model.eval()
        va_preds, va_targets, va_prob_list = [], [], []
        with torch.no_grad():
            for x, y in dl_va:
                x = x.to(device, non_blocking=True)
                p = F.softmax(model(x), dim=1)
                va_prob_list.append(p.cpu().numpy())
                va_preds.extend(p.argmax(dim=1).cpu().numpy())
                va_targets.extend(y.numpy())
        
        va_f1 = f1_score(va_targets, va_preds, average='macro')
        print(f'Epoch {ep}: Tr Loss={tr_loss/len(ds_tr):.4f} | Val Macro F1={va_f1:.5f}')
        
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = copy.deepcopy(model.state_dict())
            best_va_probs = np.vstack(va_prob_list)
    
    oof_probs[va_idx] = best_va_probs
    model.load_state_dict(best_state)
    
    # 4x TTA on Test Set for this fold
    print(f'Running 4x TTA on Test Set (Fold {fold})...')
    fold_test_probs = np.zeros((len(test_df), 5), dtype=np.float32)
    for tta_t in tta_transforms:
        ds_te = GameDataset(test_df, test_img_dir, transform=tta_t, is_test=True)
        dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        model.eval()
        tta_list = []
        with torch.no_grad():
            for x, _ in dl_te:
                p = F.softmax(model(x.to(device)), dim=1)
                tta_list.append(p.cpu().numpy())
        fold_test_probs += np.vstack(tta_list) / len(tta_transforms)
    
    test_probs += fold_test_probs / 5.0

overall_oof_f1 = f1_score(y_all, oof_probs.argmax(axis=1), average='macro')
print(f'\n🏆 Overall 5-Fold OOF Macro F1: {overall_oof_f1:.5f}')
print(classification_report(y_all, oof_probs.argmax(axis=1), target_names=CLASSES))

In [ ]:
# 7. Macro-F1 Logit Calibration & Official Submission Export
from google.colab import files

# Optimize class multipliers specifically on OOF Macro F1
best_w = np.ones(5, dtype=np.float32)
best_f1 = overall_oof_f1
for _ in range(3):
    for c in range(5):
        for cand in np.linspace(0.80, 1.25, 46):
            w_try = best_w.copy()
            w_try[c] = cand
            f1 = f1_score(y_all, (oof_probs * w_try).argmax(axis=1), average='macro')
            if f1 > best_f1:
                best_f1 = f1
                best_w[c] = cand

print(f'Calibrated OOF Macro F1: {best_f1:.5f} (multipliers={best_w.round(3)})')

# Apply multipliers to test probabilities
final_test_preds = (test_probs * best_w).argmax(axis=1)

# Official sample_submission row ordering (ANS00001 .. ANS25889)
ans_ids = [f'ANS{i:05d}' for i in range(1, len(test_df) + 1)]
sub_df = pd.DataFrame({'id': ans_ids, 'task4': final_test_preds})
sub_df.to_csv('index.csv', index=False)

print('Test Class Distribution:')
print(sub_df['task4'].value_counts().sort_index())
print('Wrote index.csv! Downloading...')
files.download('index.csv')